# Parse Legacy Pipeline Runs From Silver-Volume Logs

This notebook backfills `pipeline_runs` from existing per-run log files under the silver volume.

Current default scope is `training` only.

- Training logs must explicitly expose the original Databricks `db_run_id` before a row is backfilled.
- Verified `db_run_id` sources are the per-step logger init lines under `/silver_volume/<db_run_id>/training/...` and the final rename log that records `from: <db_run_id>` and `to: <model_run_id>`.
- Inference logs are not enabled as a default source here because the silver-volume folder is keyed by `model_run_id`, not reliably by Databricks `db_run_id`.
- `started_at` and `finished_at` are inferred from the first and last log timestamps and kept as naive datetimes from the legacy logs.
- Status is conservative: this parser marks a run `completed` only when there is explicit completion evidence. Ambiguous runs are left with `status = null` instead of being inferred as failed from generic log levels.
- Fields that are not reliably recoverable from logs alone remain `null` and parser details are stored in `payload_json`.


In [ ]:
from __future__ import annotations

import json
import pathlib
import re
import typing as t
from datetime import datetime

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession

    spark = DatabricksSession.builder.getOrCreate()

from delta.tables import DeltaTable
from pyspark.sql import functions as F

from edvise.shared.dashboard_metadata import pipeline_runs as pr

CATALOG = "staging_sst_01"
INSTITUTIONS: list[str] | None = None
RUN_TYPES = ["training"]
SCHEMA = "default"
TABLE = "pipeline_runs"
DRY_RUN = True
SKIP_INFERENCE_WITHOUT_DB_RUN_ID = True

TARGET_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"

In [ ]:
LOG_TS_RE = re.compile(
    r"^(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3}) - (?P<logger>.*?) - (?P<level>[A-Z]+) - (?P<message>.*)$"
)
RUN_PATH_RE = re.compile(
    r"/Volumes/(?P<catalog>[^/]+)/(?P<institution>[^/]+)_silver/silver_volume/(?P<run_id>[^/]+)/(?P<run_type>training|inference)/"
)
RUN_ROOT_RE = re.compile(
    r"/Volumes/(?P<catalog>[^/]+)/(?P<institution>[^/]+)_silver/silver_volume/(?P<run_id>[^/\s]+)"
)
SELECTED_TOP_RUN_RE = re.compile(
    r"Selected top run:\s+.*\s+&\s+(?P<run_id>[A-Za-z0-9_.-]+)"
)
UC_MODEL_RE = re.compile(
    r"Using UC model .* run_id=(?P<run_id>[A-Za-z0-9_.-]+), experiment_id=(?P<experiment_id>[A-Za-z0-9_.-]+)"
)
DATASET_RE = re.compile(
    r"(?P<name>(?P<kind>cohort|course)[^\s,;:]*\.csv)",
    flags=re.IGNORECASE,
)
RENAME_FROM_RE = re.compile(
    r"^\s*from:\s*(?P<path>/Volumes/[^\s]+/silver_volume/(?P<run_id>[^/\s]+))\s*$"
)
RENAME_TO_RE = re.compile(
    r"^\s*to:\s*(?P<path>/Volumes/[^\s]+/silver_volume/(?P<run_id>[^/\s]+))\s*$"
)
TRACEBACK_RE = re.compile(r"^Traceback \(most recent call last\):")


def _log_ts(raw: str) -> datetime:
    return datetime.strptime(raw, "%Y-%m-%d %H:%M:%S,%f")


def _mtime(path: pathlib.Path) -> datetime:
    return datetime.fromtimestamp(path.stat().st_mtime)


def _unique(values: t.Iterable[t.Optional[str]]) -> list[str]:
    out: list[str] = []
    seen: set[str] = set()
    for value in values:
        if value is None:
            continue
        text = str(value).strip()
        if not text or text in seen:
            continue
        seen.add(text)
        out.append(text)
    return out


def _first(values: t.Iterable[t.Optional[str]]) -> t.Optional[str]:
    vals = _unique(values)
    return vals[0] if vals else None


def discover_institutions(
    catalog: str, institutions: list[str] | None = None
) -> list[str]:
    if institutions:
        return sorted({str(inst).strip() for inst in institutions if str(inst).strip()})
    catalog_root = pathlib.Path(f"/Volumes/{catalog}")
    return sorted(
        path.name[: -len("_silver")]
        for path in catalog_root.glob("*_silver")
        if path.is_dir() and path.name.endswith("_silver")
    )


def discover_log_groups(
    catalog: str,
    institutions: list[str],
    run_types: list[str],
) -> dict[tuple[str, str, str], list[pathlib.Path]]:
    groups: dict[tuple[str, str, str], list[pathlib.Path]] = {}
    for institution_id in institutions:
        silver_root = pathlib.Path(
            f"/Volumes/{catalog}/{institution_id}_silver/silver_volume"
        )
        if not silver_root.exists():
            print(f"Skipping missing silver root: {silver_root}")
            continue
        for run_root in sorted(path for path in silver_root.iterdir() if path.is_dir()):
            for run_type in run_types:
                run_dir = run_root / run_type
                if not run_dir.exists() or not run_dir.is_dir():
                    continue
                log_paths = sorted(
                    path for path in run_dir.glob("*.log") if path.is_file()
                )
                if not log_paths:
                    continue
                groups[(institution_id, run_root.name, run_type)] = log_paths
    return groups


def show_df(df) -> None:
    try:
        display(df)
    except NameError:
        df.show(truncate=False)

In [ ]:
def parse_log_file(log_path: pathlib.Path) -> dict[str, t.Any]:
    started_at: datetime | None = None
    finished_at: datetime | None = None
    error_lines: list[str] = []
    training_db_run_ids: list[str] = []
    rename_from_run_ids: list[str] = []
    rename_to_run_ids: list[str] = []
    model_run_ids: list[str] = []
    experiment_ids: list[str] = []
    cohort_dataset_names: list[str] = []
    course_dataset_names: list[str] = []
    completion_markers: list[str] = []

    for raw_line in log_path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = raw_line.rstrip("\n")
        matched = LOG_TS_RE.match(line)
        if matched:
            ts = _log_ts(matched.group("ts"))
            started_at = ts if started_at is None or ts < started_at else started_at
            finished_at = ts if finished_at is None or ts > finished_at else finished_at

            level = matched.group("level")
            message = matched.group("message")

            run_path_match = RUN_PATH_RE.search(message)
            if run_path_match:
                gd = run_path_match.groupdict()
                if gd["run_type"] == "training":
                    training_db_run_ids.append(gd["run_id"])
                else:
                    model_run_ids.append(gd["run_id"])

            selected_top_run_match = SELECTED_TOP_RUN_RE.search(message)
            if selected_top_run_match:
                model_run_ids.append(selected_top_run_match.group("run_id"))

            uc_model_match = UC_MODEL_RE.search(message)
            if uc_model_match:
                model_run_ids.append(uc_model_match.group("run_id"))
                experiment_ids.append(uc_model_match.group("experiment_id"))

            if (
                "Updating folder name to model id" in message
                or "Renaming run root:" in message
            ):
                completion_markers.append(message)

            for dataset_match in DATASET_RE.finditer(message):
                dataset_name = dataset_match.group("name")
                if dataset_match.group("kind").lower() == "cohort":
                    cohort_dataset_names.append(dataset_name)
                else:
                    course_dataset_names.append(dataset_name)

            continue

        rename_from_match = RENAME_FROM_RE.match(line)
        if rename_from_match:
            rename_from_run_ids.append(rename_from_match.group("run_id"))
            completion_markers.append(line)

        rename_to_match = RENAME_TO_RE.match(line)
        if rename_to_match:
            rename_to_run_ids.append(rename_to_match.group("run_id"))
            completion_markers.append(line)

        for dataset_match in DATASET_RE.finditer(line):
            dataset_name = dataset_match.group("name")
            if dataset_match.group("kind").lower() == "cohort":
                cohort_dataset_names.append(dataset_name)
            else:
                course_dataset_names.append(dataset_name)

        if TRACEBACK_RE.match(line):
            error_lines.append(line)

    fallback_ts = _mtime(log_path)
    return {
        "log_path": str(log_path),
        "started_at": started_at,
        "finished_at": finished_at,
        "fallback_ts": fallback_ts,
        "training_db_run_ids": _unique(training_db_run_ids),
        "rename_from_run_ids": _unique(rename_from_run_ids),
        "rename_to_run_ids": _unique(rename_to_run_ids),
        "model_run_ids": _unique(model_run_ids),
        "experiment_ids": _unique(experiment_ids),
        "cohort_dataset_names": _unique(cohort_dataset_names),
        "course_dataset_names": _unique(course_dataset_names),
        "completion_markers": _unique(completion_markers),
        "error_lines": _unique(error_lines),
    }


def aggregate_run_logs(
    institution_id: str,
    run_root_id: str,
    run_type: str,
    log_paths: list[pathlib.Path],
) -> tuple[dict[str, t.Any] | None, dict[str, t.Any] | None]:
    parsed_logs = [parse_log_file(path) for path in sorted(log_paths)]

    started_candidates = [
        p["started_at"] for p in parsed_logs if p["started_at"] is not None
    ]
    finished_candidates = [
        p["finished_at"] for p in parsed_logs if p["finished_at"] is not None
    ]
    fallback_candidates = [p["fallback_ts"] for p in parsed_logs]

    started_at = (
        min(started_candidates) if started_candidates else min(fallback_candidates)
    )
    finished_at = (
        max(finished_candidates) if finished_candidates else max(fallback_candidates)
    )
    updated_at = finished_at or started_at

    training_db_run_id_candidates = _unique(
        candidate
        for parsed_log in parsed_logs
        for candidate in (
            parsed_log["training_db_run_ids"] + parsed_log["rename_from_run_ids"]
        )
    )
    if len(training_db_run_id_candidates) != 1:
        return None, {
            "institution_id": institution_id,
            "run_root_id": run_root_id,
            "run_type": run_type,
            "reason": "db_run_id could not be recovered uniquely from logs",
            "db_run_id_candidates": training_db_run_id_candidates,
            "log_paths": [str(path) for path in log_paths],
        }

    training_db_run_id = training_db_run_id_candidates[0]
    explicit_model_run_id = _first(
        candidate
        for parsed_log in parsed_logs
        for candidate in parsed_log["model_run_ids"]
    )
    experiment_id = _first(
        candidate
        for parsed_log in parsed_logs
        for candidate in parsed_log["experiment_ids"]
    )
    cohort_dataset_name = _first(
        candidate
        for parsed_log in parsed_logs
        for candidate in parsed_log["cohort_dataset_names"]
    )
    course_dataset_name = _first(
        candidate
        for parsed_log in parsed_logs
        for candidate in parsed_log["course_dataset_names"]
    )
    error_message = _first(
        candidate
        for parsed_log in reversed(parsed_logs)
        for candidate in parsed_log["error_lines"]
    )
    rename_to_run_ids = _unique(
        candidate
        for parsed_log in parsed_logs
        for candidate in parsed_log["rename_to_run_ids"]
    )
    completion_markers = _unique(
        candidate
        for parsed_log in parsed_logs
        for candidate in parsed_log["completion_markers"]
    )

    notes: list[str] = []

    if run_type == "training":
        run_id = training_db_run_id
        run_id_source = "logged_training_path_or_rename_from"
    else:
        run_id = training_db_run_id
        run_id_source = "logged_training_path_or_rename_from"
        if not run_id and SKIP_INFERENCE_WITHOUT_DB_RUN_ID:
            return None, {
                "institution_id": institution_id,
                "run_root_id": run_root_id,
                "run_type": run_type,
                "reason": "db_run_id could not be recovered from logs",
                "log_paths": [str(path) for path in log_paths],
            }
        if not run_id:
            return None, {
                "institution_id": institution_id,
                "run_root_id": run_root_id,
                "run_type": run_type,
                "reason": "db_run_id could not be recovered from logs",
                "log_paths": [str(path) for path in log_paths],
            }

    model_run_id = explicit_model_run_id
    model_run_id_source = None
    if model_run_id:
        model_run_id_source = "log_message"
    elif run_type == "training" and run_root_id != run_id:
        model_run_id = run_root_id
        model_run_id_source = "run_root"
    elif run_type == "inference":
        model_run_id = run_root_id
        model_run_id_source = "run_root"

    dataset_ts = pr.parse_timestamp_from_filename(
        cohort_dataset_name
    ) or pr.parse_timestamp_from_filename(course_dataset_name)
    completion_detected = False
    status = None
    status_source = None
    if run_type == "training":
        if run_root_id != training_db_run_id:
            completion_detected = True
            status = "completed"
            status_source = "renamed_run_root"
        if run_root_id in rename_to_run_ids:
            completion_detected = True
            status = "completed"
            status_source = "rename_to_log"
    elif completion_markers:
        completion_detected = True
        status = "completed"
        status_source = "completion_marker"

    payload = {
        "legacy_backfill": True,
        "parser": "parse_legacy_pipeline_runs.ipynb",
        "log_paths": [parsed_log["log_path"] for parsed_log in parsed_logs],
        "run_root_id": run_root_id,
        "run_id_source": run_id_source,
        "model_run_id_source": model_run_id_source,
        "completion_detected": completion_detected,
        "status_source": status_source,
    }
    if notes:
        payload["notes"] = notes
    if error_message:
        payload["error_excerpt"] = error_message

    row = {
        "institution_id": institution_id,
        "run_id": str(run_id),
        "run_type": run_type,
        "status": status,
        "started_at": started_at,
        "finished_at": finished_at,
        "updated_at": updated_at,
        "run_url": None,
        "cohort_dataset_name": cohort_dataset_name,
        "course_dataset_name": course_dataset_name,
        "dataset_ts": dataset_ts,
        "cohort": None,
        "term_filter": None,
        "model_run_id": model_run_id,
        "experiment_id": experiment_id,
        "pipeline_version": None,
        "error_message": error_message,
        "payload_json": json.dumps(payload, sort_keys=True),
    }
    return row, None


def build_legacy_run_rows(
    catalog: str,
    institutions: list[str] | None = None,
    run_types: list[str] | None = None,
) -> tuple[list[dict[str, t.Any]], list[dict[str, t.Any]]]:
    institutions2 = discover_institutions(catalog, institutions)
    run_types2 = run_types or ["training"]
    groups = discover_log_groups(catalog, institutions2, run_types2)

    rows: list[dict[str, t.Any]] = []
    skipped: list[dict[str, t.Any]] = []
    for (institution_id, run_root_id, run_type), log_paths in sorted(groups.items()):
        row, skip = aggregate_run_logs(institution_id, run_root_id, run_type, log_paths)
        if row is not None:
            rows.append(row)
        if skip is not None:
            skipped.append(skip)
    return rows, skipped

In [ ]:
rows, skipped = build_legacy_run_rows(
    catalog=CATALOG,
    institutions=INSTITUTIONS,
    run_types=RUN_TYPES,
)

print(f"Parsed rows: {len(rows)}")
print(f"Skipped groups: {len(skipped)}")

schema = pr._get_pipeline_runs_schema()
if not rows:
    parsed_df = spark.createDataFrame([], schema=schema)
else:
    parsed_df = spark.createDataFrame(rows, schema=schema)

show_df(parsed_df.orderBy("institution_id", "run_type", "started_at"))
show_df(
    parsed_df.groupBy("institution_id", "run_type", "status")
    .count()
    .orderBy("institution_id", "run_type", "status")
)

if skipped:
    skipped_df = spark.createDataFrame(skipped)
    show_df(skipped_df.orderBy("institution_id", "run_type", "run_root_id"))

In [ ]:
def upsert_pipeline_runs_df(df, table_path: str) -> None:
    try:
        dt = DeltaTable.forName(spark, table_path)
    except Exception:
        (
            df.write.format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(table_path)
        )
        return

    finished_at_expr = (
        F.when(F.col("s.finished_at").isNull(), F.col("t.finished_at"))
        .when(F.col("t.finished_at").isNull(), F.col("s.finished_at"))
        .when(F.col("s.finished_at") > F.col("t.finished_at"), F.col("s.finished_at"))
        .otherwise(F.col("t.finished_at"))
    )
    status_expr = F.expr(
        """
        CASE
          WHEN t.status = 'failed' OR s.status = 'failed' THEN 'failed'
          WHEN t.status = 'completed' OR s.status = 'completed' THEN 'completed'
          WHEN t.status = 'running' OR s.status = 'running' THEN 'running'
          ELSE COALESCE(s.status, t.status)
        END
        """
    )

    (
        dt.alias("t")
        .merge(df.alias("s"), "t.run_id = s.run_id AND t.run_type = s.run_type")
        .whenMatchedUpdate(
            set={
                "updated_at": F.col("s.updated_at"),
                "run_url": F.coalesce(F.col("s.run_url"), F.col("t.run_url")),
                "institution_id": F.coalesce(
                    F.col("s.institution_id"), F.col("t.institution_id")
                ),
                "status": status_expr,
                "started_at": F.coalesce(F.col("t.started_at"), F.col("s.started_at")),
                "finished_at": finished_at_expr,
                "cohort_dataset_name": F.coalesce(
                    F.col("s.cohort_dataset_name"), F.col("t.cohort_dataset_name")
                ),
                "course_dataset_name": F.coalesce(
                    F.col("s.course_dataset_name"), F.col("t.course_dataset_name")
                ),
                "dataset_ts": F.coalesce(F.col("s.dataset_ts"), F.col("t.dataset_ts")),
                "cohort": F.coalesce(F.col("s.cohort"), F.col("t.cohort")),
                "term_filter": F.coalesce(
                    F.col("s.term_filter"), F.col("t.term_filter")
                ),
                "model_run_id": F.coalesce(
                    F.col("s.model_run_id"), F.col("t.model_run_id")
                ),
                "experiment_id": F.coalesce(
                    F.col("s.experiment_id"), F.col("t.experiment_id")
                ),
                "pipeline_version": F.coalesce(
                    F.col("s.pipeline_version"), F.col("t.pipeline_version")
                ),
                "error_message": F.coalesce(
                    F.col("s.error_message"), F.col("t.error_message")
                ),
                "payload_json": F.coalesce(
                    F.col("s.payload_json"), F.col("t.payload_json")
                ),
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [ ]:
if DRY_RUN:
    print(
        f"DRY_RUN=True. Parsed {len(rows)} rows and skipped {len(skipped)} groups. No data was written to {TARGET_TABLE}."
    )
else:
    upsert_pipeline_runs_df(parsed_df, TARGET_TABLE)
    print(f"Merged {len(rows)} rows into {TARGET_TABLE}.")